# 04 — Inference

Generates text from a trained checkpoint.

- **No args** → picks up the most recently modified training run automatically.
- **Explicit config** → use `configs/infer_exp001.json` (supports `$from_run` injection).

Outputs are written to `runs/infer/<infer_id>/generations.jsonl`.

In [ ]:
# Cell 1 — project root
import os

# os.chdir("/notebooks/alg3")                        # Nebius / RunPod / cloud
os.chdir("/Users/dmitrit/Documents/dev/alg3")       # local

print("Working directory:", os.getcwd())

In [ ]:
# Cell 2 — install dependencies (skip if already installed)
# !pip install torch tokenizers tensorboard numpy --quiet

In [ ]:
# Cell 3 — quick inference: latest training run, default prompt
import sys
sys.path.insert(0, "src")

from infer import run_inference

run_inference()   # no args → auto-finds latest run

In [ ]:
# Cell 4 — explicit inference config (with custom prompts)
# Edit configs/infer_exp001.json to set your prompts and generation params,
# then run this cell.

run_inference("configs/infer_exp001.json")

In [ ]:
# Cell 5 — interactive: generate from a custom prompt inline
import sys, json, torch
from pathlib import Path
sys.path.insert(0, "src")

from model import GPT
from tokenizer import Tokenizer

# ── settings ────────────────────────────────────────────
PROMPT         = "Once upon a time"   # change me
MAX_NEW_TOKENS = 300
TEMPERATURE    = 0.9
TOP_K          = 50
# ────────────────────────────────────────────────────────

# Auto-detect device
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")

# Load latest checkpoint
run_dir  = sorted(Path("runs/train").iterdir(), key=lambda p: p.stat().st_mtime)[-1]
ckpt     = run_dir / "checkpoints" / "best.pt"
if not ckpt.exists():
    ckpt = run_dir / "checkpoints" / "latest.pt"

resolved = json.loads((run_dir / "resolved_run.json").read_text())
tok_id   = resolved["tokenizer"]["tok_id"]

model, _ = GPT.from_checkpoint(str(ckpt))
model    = model.to(device).eval()
tok      = Tokenizer.load(tok_id)

print(f"Run      : {run_dir.name}")
print(f"Checkpoint: {ckpt.name}")
print(f"Device   : {device}")
print(f"Params   : {model.num_parameters()/1e6:.1f}M")
print(f"Prompt   : {PROMPT!r}")
print("=" * 70)

ids = tok.encode(PROMPT)
idx = torch.tensor([ids], dtype=torch.long, device=device)
with torch.no_grad():
    out = model.generate(idx, MAX_NEW_TOKENS, temperature=TEMPERATURE, top_k=TOP_K)

print(tok.decode(out[0].tolist()))

In [ ]:
# Cell 6 — show all past inference runs
import json
from pathlib import Path

infer_runs = sorted(Path("runs/infer").iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
for run_dir in infer_runs[:5]:   # show 5 most recent
    mf = run_dir / "manifest.json"
    if not mf.exists():
        continue
    m = json.loads(mf.read_text())
    print(f"{m['run_id']}  status={m['status']}")
    gen_path = run_dir / "generations.jsonl"
    if gen_path.exists():
        with open(gen_path) as f:
            lines = f.readlines()
        print(f"  {len(lines)} generation(s)")
        first = json.loads(lines[0])
        preview = first["generated"][:120].replace("\n", " ")
        print(f"  {preview!r}...")
    print()